# Tugas FCFS and SRT

- **NIM**   : 2802451876
- **Nama**  : Albertus Adrian

Arrival Time dari NIM `2802451876` (digit `0` diganti `8`).

Burst Time dari `6781542082` (digit `0` diganti `8`).

---

In [7]:
from dataclasses import dataclass


@dataclass
class Proc:
    name: str
    arrival: int
    burst: int
    order: int
    ct: int = 0
    tat: int = 0
    wt: int = 0


def print_table(title, procs):
    print(f"\n=== {title} ===")
    print(f"{'Proc':<6}{'Arrival':<9}{'Burst':<8}{'Complete':<11}{'TAT':<5}{'WT':<5}")
    sum_tat = 0
    sum_wt = 0
    for x in procs:
        print(f"{x.name:<6}{x.arrival:<9}{x.burst:<8}{x.ct:<11}{x.tat:<5}{x.wt:<5}")
        sum_tat += x.tat
        sum_wt += x.wt
    n = len(procs)
    print(f"Average Turnaround Time : {sum_tat / n:.2f}")
    print(f"Average Waiting Time    : {sum_wt / n:.2f}")


def gantt(title, procs):
    print(f"\nGantt Chart ({title}):")
    print("|" + "|".join(f" {x.name} " for x in procs) + "|")
    w = len(procs[0].name) + 3
    print(f"{procs[0].ct - procs[0].burst:<{w}}", end="")
    for x in procs:
        print(f"{x.ct:<{len(x.name) + 3}}", end="")
    print()


def print_timeline(title, timeline):
    print(f"\n=== Timeline {title} ===")
    headers = ["Waktu", "Proses", "Sisa BT sblm proc", "Arr", "Burst", "Complete", "TAT", "WT"]
    print(f"{headers[0]:<10}{headers[1]:<8}{headers[2]:<20}{headers[3]:<6}{headers[4]:<7}{headers[5]:<10}{headers[6]:<6}{headers[7]:<6}")
    for row in timeline:
        print(f"{row['waktu']:<10}{row['proses']:<8}{row['sisa_sebelum']:<20}{row['arrival']:<6}{row['burst']:<7}{str(row['complete']):<10}{str(row['tat']):<6}{str(row['wt']):<6}")

In [8]:
def fcfs(procs):
    ordered = sorted(procs, key=lambda x: (x.arrival, x.order))
    timeline = []
    time = 0

    for proc in ordered:
        if time < proc.arrival:
            timeline.append({"waktu": f"{time}-{proc.arrival}", "proses": "Idle", "sisa_sebelum": "-", "arrival": "-", "burst": "-", "complete": "-", "tat": "-", "wt": "-"})
            time = proc.arrival
        start = time
        proc.ct = time + proc.burst
        proc.tat = proc.ct - proc.arrival
        proc.wt = proc.tat - proc.burst
        timeline.append({"waktu": f"{start}-{proc.ct}", "proses": proc.name, "sisa_sebelum": proc.burst, "arrival": proc.arrival, "burst": proc.burst, "complete": proc.ct, "tat": proc.tat, "wt": proc.wt})
        time = proc.ct
    return ordered, timeline


In [9]:
arrival = [2, 8, 8, 2, 4, 5, 1, 8, 7, 6]
burst = [6, 7, 8, 1, 5, 4, 2, 8, 8, 2]

procs = [Proc(f"P{i + 1}", arrival[i], burst[i], i) for i in range(10)]

print("NIM   : 2802451876")
print("Nama  : Albertus Adrian")
print("CPU Scheduling: FCFS and SRT")

NIM   : 2802451876
Nama  : Albertus Adrian
CPU Scheduling: FCFS and SRT


---
## 1. FCFS (First Come First Serve)

In [10]:
f, fcfs_timeline = fcfs([Proc(x.name, x.arrival, x.burst, x.order) for x in procs])
print_timeline("FCFS", fcfs_timeline)
gantt("FCFS", f)
print_table("FCFS", f)


=== Timeline FCFS ===
Waktu     Proses  Sisa BT sblm proc   Arr   Burst  Complete  TAT   WT    
0-1       Idle    -                   -     -      -         -     -     
1-3       P7      2                   1     2      3         2     0     
3-9       P1      6                   2     6      9         7     1     
9-10      P4      1                   2     1      10        8     7     
10-15     P5      5                   4     5      15        11    6     
15-19     P6      4                   5     4      19        14    10    
19-21     P10     2                   6     2      21        15    13    
21-29     P9      8                   7     8      29        22    14    
29-36     P2      7                   8     7      36        28    21    
36-44     P3      8                   8     8      44        36    28    
44-52     P8      8                   8     8      52        44    36    

Gantt Chart (FCFS):
| P7 | P1 | P4 | P5 | P6 | P10 | P9 | P2 | P3 | P8 |
1    3    9    

---
## 2. SRT (Shortest Remaining Time - preemptive)

In [11]:
def srt(procs):
    remaining = {proc.name: proc.burst for proc in procs}
    timeline = []
    time = 0
    completed = 0

    while completed < len(procs):
        ready = [proc for proc in procs if proc.arrival <= time and remaining[proc.name] > 0]
        if not ready:
            next_arrival = min(proc.arrival for proc in procs if remaining[proc.name] > 0)
            timeline.append({"waktu": f"{time}-{next_arrival}", "proses": "Idle", "sisa_sebelum": "-", "arrival": "-", "burst": "-", "complete": "-", "tat": "-", "wt": "-"})
            time = next_arrival
            continue

        proc = min(ready, key=lambda x: (remaining[x.name], x.arrival, x.order))
        before = remaining[proc.name]
        time += 1
        remaining[proc.name] -= 1
        just_completed = remaining[proc.name] == 0
        if just_completed:
            proc.ct = time
            proc.tat = proc.ct - proc.arrival
            proc.wt = proc.tat - proc.burst
            completed += 1

        timeline.append({"waktu": f"{time - 1}-{time}", "proses": proc.name, "sisa_sebelum": before, "arrival": proc.arrival, "burst": proc.burst, "complete": proc.ct if just_completed else "-", "tat": proc.tat if just_completed else "-", "wt": proc.wt if just_completed else "-"})

    return sorted(procs, key=lambda x: (x.ct, x.order)), timeline

In [12]:
s, srt_timeline = srt([Proc(x.name, x.arrival, x.burst, x.order) for x in procs])
print_timeline("SRT", srt_timeline)
print_table("SRT", s)


=== Timeline SRT ===
Waktu     Proses  Sisa BT sblm proc   Arr   Burst  Complete  TAT   WT    
0-1       Idle    -                   -     -      -         -     -     
1-2       P7      2                   1     2      -         -     -     
2-3       P7      1                   1     2      3         2     0     
3-4       P4      1                   2     1      4         2     1     
4-5       P5      5                   4     5      -         -     -     
5-6       P5      4                   4     5      -         -     -     
6-7       P10     2                   6     2      -         -     -     
7-8       P10     1                   6     2      8         2     0     
8-9       P5      3                   4     5      -         -     -     
9-10      P5      2                   4     5      -         -     -     
10-11     P5      1                   4     5      11        7     2     
11-12     P6      4                   5     4      -         -     -     
12-13     P6    